In [ ]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数（演示：带未来函数 / 前视偏差的极简因子）

    评测时平台会自动替换 datasources / start_date / end_date 三个入参并调用本函数

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}
                                "bar1m"     -> 分钟 K 线表
                                "financial" -> 财务数据表
        start_date (str): 开始时间
        end_date (str):   结束时间

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 用未来函数，需要向后多取若干天，才能算出区间末尾几天的"明日收益"
    LOOKAHEAD_DAYS = 7
    query_end_date = pd.to_datetime(end_date) + pd.Timedelta(days=LOOKAHEAD_DAYS)

    # ===== 编写因子 SQL =====
    # 【未来函数示例】把"明日日收益率"直接当作"今日因子值"。
    # 由于评测用的是 T+1 收益，因子几乎等于被预测目标本身，IC 会异常地高（接近 1），
    # 这正是典型的前视偏差（look-ahead bias），仅用于演示，切勿用于实盘/正式提交。
    sql = f"""
    WITH cte_daily AS (
        -- 分钟 K 线聚合成日频收盘价（取每个交易日最后一分钟的 close）
        SELECT
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            last(close ORDER BY date)  AS close
        FROM {bar1m}
        WHERE close > 0
        GROUP BY instrument, strftime(date, '%Y-%m-%d')
    ),
    cte_lead AS (
        SELECT
            *,
            -- 关键的未来函数：lead 取到"下一交易日"的收盘价
            lead(close, 1) OVER (PARTITION BY instrument ORDER BY trading_day) AS next_close
        FROM cte_daily
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        -- 用明日收益率作为今日因子（未来函数）
        (next_close / close - 1) AS factor
    FROM cte_lead
    WHERE next_close IS NOT NULL
    """

    # 注意：因为要用到 end_date 之后的数据算"明日收益"，这里把查询上界放宽到 query_end_date，
    # 最终再用 start_date ~ end_date 过滤输出。
    df = dai.query(
        sql,
        filters={'date': [start_date, query_end_date.strftime('%Y-%m-%d %H:%M:%S')]},
        compression=True,
    ).df()
    df = df[(df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))]

    # ===== 对齐股票池 =====
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    return df


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
